<a href="https://colab.research.google.com/github/AmalMohammed95/SARA-Smart-Academic-Research-Agent/blob/main/SARA_Smart_Academic_Research_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# SARA - Smart Academic Research Agent
# Project 02 - Elicit
# Level 2 - Applied Agent

print("SARA environment is ready.")

SARA environment is ready.


In [3]:
!pip -q install requests pandas

In [4]:
import requests
import pandas as pd
import json
import sqlite3
import time

print("All basic libraries loaded successfully.")

All basic libraries loaded successfully.


In [5]:
# SARA Agent State

sara_state = {
    "research_question": "",
    "sub_questions": [],
    "search_keywords": [],
    "retrieved_papers": [],
    "selected_papers": [],
    "excluded_papers": [],
    "exclusion_reasons": {},
    "extracted_evidence": [],
    "identified_gaps": [],
    "iteration": 0,
    "stopping_reason": None,
    "execution_log": []
}

print("SARA state initialized successfully.")
print(sara_state)

SARA state initialized successfully.
{'research_question': '', 'sub_questions': [], 'search_keywords': [], 'retrieved_papers': [], 'selected_papers': [], 'excluded_papers': [], 'exclusion_reasons': {}, 'extracted_evidence': [], 'identified_gaps': [], 'iteration': 0, 'stopping_reason': None, 'execution_log': []}


In [6]:
# Step 4 - Set Research Question

research_question = "What are the recent applications of agentic AI in academic research and literature review?"

sara_state["research_question"] = research_question

sara_state["sub_questions"] = [
    "What is agentic AI in the context of academic research?",
    "How is agentic AI used in literature review workflows?",
    "What tools or frameworks are commonly used?",
    "What are the main benefits and limitations?"
]

sara_state["search_keywords"] = [
    "agentic AI academic research",
    "agentic AI literature review",
    "AI agents scholarly research",
    "autonomous agents literature review"
]

sara_state["execution_log"].append({
    "step": "research_planning",
    "status": "completed",
    "details": "Research question decomposed into sub-questions and search keywords."
})

print("Research plan created successfully.\n")

print("Research Question:")
print(sara_state["research_question"])

print("\nSub-Questions:")
for i, q in enumerate(sara_state["sub_questions"], 1):
    print(f"{i}. {q}")

print("\nSearch Keywords:")
for i, k in enumerate(sara_state["search_keywords"], 1):
    print(f"{i}. {k}")

Research plan created successfully.

Research Question:
What are the recent applications of agentic AI in academic research and literature review?

Sub-Questions:
1. What is agentic AI in the context of academic research?
2. How is agentic AI used in literature review workflows?
3. What tools or frameworks are commonly used?
4. What are the main benefits and limitations?

Search Keywords:
1. agentic AI academic research
2. agentic AI literature review
3. AI agents scholarly research
4. autonomous agents literature review


In [7]:
# Step 5 - Search OpenAlex

def search_openalex(query, per_page=5):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "filter": "from_publication_date:2021-01-01,language:en",
        "per-page": per_page
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    papers = []

    for work in data.get("results", []):
        paper = {
            "title": work.get("title"),
            "year": work.get("publication_year"),
            "doi": work.get("doi"),
            "openalex_id": work.get("id"),
            "cited_by_count": work.get("cited_by_count", 0)
        }
        papers.append(paper)

    return papers


# Test the search tool
test_query = sara_state["search_keywords"][0]

papers = search_openalex(test_query)

print(f"Search query: {test_query}")
print(f"Papers retrieved: {len(papers)}\n")

for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper['title']}")
    print(f"   Year: {paper['year']}")
    print(f"   DOI: {paper['doi']}")
    print(f"   Citations: {paper['cited_by_count']}")
    print()

Search query: agentic AI academic research
Papers retrieved: 5

1. Opinion Paper: “So what if ChatGPT wrote it?” Multidisciplinary perspectives on opportunities, challenges and implications of generative conversational AI for research, practice and policy
   Year: 2023
   DOI: https://doi.org/10.1016/j.ijinfomgt.2023.102642
   Citations: 4323

2. Conceptualizing AI literacy: An exploratory review
   Year: 2021
   DOI: https://doi.org/10.1016/j.caeai.2021.100041
   Citations: 1899

3. Performance of ChatGPT on USMLE: Potential for AI-assisted medical education using large language models
   Year: 2023
   DOI: https://doi.org/10.1371/journal.pdig.0000198
   Citations: 3842

4. Generative AI
   Year: 2023
   DOI: https://doi.org/10.1007/s12599-023-00834-7
   Citations: 1364

5. Students’ voices on generative AI: perceptions, benefits, and challenges in higher education
   Year: 2023
   DOI: https://doi.org/10.1186/s41239-023-00411-8
   Citations: 2206



In [8]:
# Test OpenAlex connection

test_results = search_openalex(
    "agentic AI academic research",
    per_page=3
)

print("OpenAlex connection successful.")
print("Papers retrieved:", len(test_results))

for i, paper in enumerate(test_results, 1):
    print(f"{i}. {paper['title']}")

OpenAlex connection successful.
Papers retrieved: 3
1. Opinion Paper: “So what if ChatGPT wrote it?” Multidisciplinary perspectives on opportunities, challenges and implications of generative conversational AI for research, practice and policy
2. Conceptualizing AI literacy: An exploratory review
3. Performance of ChatGPT on USMLE: Potential for AI-assisted medical education using large language models


In [9]:
# Step 6 - Search using all SARA keywords

all_retrieved_papers = []

for query in sara_state["search_keywords"]:
    print(f"Searching: {query}")

    results = search_openalex(query, per_page=10)

    for paper in results:
        paper["search_query"] = query
        all_retrieved_papers.append(paper)

    print(f"Retrieved: {len(results)} papers\n")


# Remove duplicates using DOI or OpenAlex ID
unique_papers = {}

for paper in all_retrieved_papers:
    key = paper["doi"] if paper["doi"] else paper["openalex_id"]

    if key not in unique_papers:
        unique_papers[key] = paper


sara_state["retrieved_papers"] = list(unique_papers.values())

sara_state["execution_log"].append({
    "step": "academic_search",
    "status": "completed",
    "details": f"{len(sara_state['retrieved_papers'])} unique papers retrieved from OpenAlex."
})


print("Search completed.")
print(f"Total retrieved before deduplication: {len(all_retrieved_papers)}")
print(f"Unique papers after deduplication: {len(sara_state['retrieved_papers'])}")

Searching: agentic AI academic research
Retrieved: 10 papers

Searching: agentic AI literature review
Retrieved: 10 papers

Searching: AI agents scholarly research
Retrieved: 10 papers

Searching: autonomous agents literature review
Retrieved: 10 papers

Search completed.
Total retrieved before deduplication: 40
Unique papers after deduplication: 32
